# Apply and Detect SynthID

Matt Hall, Bergen

---

**NB This notebook uses a local model with about 1.7 billion parameters. Loading this as 32-bit floats will cost nearly 7GB of RAM.**

Because the model is local on your machine, you may use any kind of data in your prompts; but if you use anything other than OPEN data, be careful how and where you share this notebook.

Google published the SynthID-Text method (Dathathri et al, 2024), building on an algorithm published the previous year by researchers at the University of Maryland (Kirchenbauer et al, 2023). The chief improvement was adding a more robust way (the 'tournament') to choose the 'winning' token at each step. In their paper, the Google researchers state that the algorithm was already in production in their Gemini web app in October 2024 (although [apparently](https://discuss.ai.google.dev/t/does-gemini-api-text-output-carry-synthid-watermarking-gemini-2-5-flash-lite-gemini-3-1-flash-lite-eu-ai-act-art-50-2/177241) not on the API). More recently, in August 2026, Anthropic [announced](https://www.anthropic.com/news/claude-text-watermark) that — in accordance with new EU regulations — they would begin watermarking Claude's output.

There is a nice explainer of the method here: https://declaude.org/watermarking/

The generation step in this notebook follows the method on the Hugging Face blog: https://huggingface.co/blog/synthid-text

#### References

Kirchenbauer, John, Jonas Geiping, Yuxin Wen, Jonathan Katz, Ian Miers and Tom Goldstein (2023). A watermark for large language models. https://arxiv.org/pdf/2301.10226

Dathathri, S, A See, S Ghaisas, et al (2024). Scalable watermarking for identifying large language model outputs. Nature 634, 818–823. https://doi.org/10.1038/s41586-024-08025-4

## Contents

The **bold sections** are probably the only bits you need.

- **Configure the LLM**
- **Generate watermarked text**
- **Generate unwatermarked text**
- **Detecting the watermark**
- Visualize the affected tokens
- Detection fails with the wrong watermarking config
- p-value varies with content length
- Code generation needs more tokens for detection
- Detection improves with greater key length

## Configure the LLM and watermarking

I was using Google's Gemma-2 2B (`google/gemma-2-2b-it`) but it is 'gated' so needs a Hugging Face token, as well as approving a license agreement — but we can use any open weight model so I have switched to Hugging Face's own SmolLM2 1.7B. It will be cached on your machine.

Instantiate the tokenizer:

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id= "HuggingFaceTB/SmolLM2-1.7B-Instruct"
TOKENIZER = AutoTokenizer.from_pretrained(model_id)

Instantiate the language model itself; this will be slow the first time you do it.

In [2]:
MODEL = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16)

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Now we can configure the watermarking with a set of integer keys and a 'window' length. These two things are combined to deterministically (i.e. repeatably, therefore detectably) generate pseudorandom sequences like [1, 0, 0] or [1, 1, 0], which are then used to weight the selection of tokens as they are generated.

In [3]:
from transformers import SynthIDTextWatermarkingConfig

watermarking_config = SynthIDTextWatermarkingConfig(
    keys=[654, 321, 987],   # Secret keys, usually length 8-16.
    ngram_len=5,            # Context length for RNG, usually 5.
)

## Generate watermarked text

We tokenize the input as we normally would without watermarking. This can be straightforward, but there is a little engineering to do for the SmolLM model.

Some models, eg `google/gemma-2-2b-it`, can consume ordinary prompt strings like so:

    TOKENIZER(prompt, return_tensors="pt")
    
The model then appends to this in generation (i.e. they just continue their input). But others, like SmolLM, need `messages`. Rather than trying to adapt our code, we let `apply_chat_template` take care of the difference:

In [4]:
prompt = "Explain generative text watermarking in simple terms."

def make_inputs(prompt: str) -> dict:
    """
    Wrap message construction and `apply_chat_template` together.
    """
    messages = [{"role": "user", "content": prompt}]
    inputs = TOKENIZER.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    )
    return inputs.to(MODEL.device)

inputs = make_inputs(prompt)
inputs

{'input_ids': tensor([[    1,  9690,   198,  2683,   359,   253,  5356,  5646, 11173,  3365,
          3511,   308, 34519,    28,  7018,   411,   407, 19712,  8182,     2,
           198,     1,  4093,   198, 36971, 44097,  1694,   913, 36671,   281,
          2232,  2656,    30,     2,   198,     1,   520,  9531,   198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

Now pass the inputs to the generation function — along with the watermarking config:

In [5]:
def make_response(**kwargs) -> str:
    """
    Send the inputs to the model, retrieve the output tokens,
    and return the decoded text.
    """
    output_ids = MODEL.generate(**kwargs, do_sample=True)
    new_tokens = output_ids[0][kwargs["input_ids"].shape[1]:]
    text = TOKENIZER.decode(new_tokens, skip_special_tokens=True)
    return text

watermarked_text = make_response(
    **inputs,
    max_new_tokens=400,
    watermarking_config=watermarking_config
)

print(watermarked_text)

Generative text watermarking is a complex AI technique which I'll try to explain in simple terms.

Imagine you're writing a very long story, and you want to put a small, secret message in it without the reader noticing. Here's a step-by-step guide on how you might do it using generative text watermarking:

1. **Choose a Technique**: There are several techniques to watermark text, but one of the simplest ones is called 'Hidden Text' or 'Hidden Text Watermarking'. This technique involves injecting a small, easily visible "marker" of text that is non-sensical in the main text but can be 'uncovered' by experts. 

2. **Identify Text Phrases**: Some good phrases for hiding text are keywords like "HIDDEN", "HW1", "HW2", etc. These are the phrases that will be used for watermarking. 

3. **Add Watermarked Phrases**: Write these hidden watermark phrases where they aren't easily noticeable, like at the end of the sentence or paragraph. The idea here isn't to hide them by obfuscating meaning, but

## Generate unwatermarked text

Let's compare to unwatermarked text.

In [6]:
unwatermarked_text = make_response(
    **inputs,
    max_new_tokens=400,
)

print(unwatermarked_text)

Story in Simple Terms
Generative text watermarking is a method of creating a unique, personalized piece of text that can be embedded within a larger piece of content. It's like a digital signature or a fingerprint, used to verify the authenticity of the content. This watermark is created using an AI or machine learning model, which can generate a unique piece of text based on the context or content of the larger piece.

How It Works
Here's a simple example of how generative text watermarking works:

1. The AI models on the platform analyze the content and understand its context.
2. The AI generates a unique, personalized text based on this analysis.
3. This text is embedded within the original content.
4. The embedded text can be seen only by an authorized user or program that has the watermark set up.

Watermarked content is like wearing a digital badge – it tells others that this content belongs to you or was created by your team, even if the physical product is different.

Applicati

The text looks different, but nearly all of the difference is normal variance between instances of the output. 

💡 To try to make the texts closer, we could take the first few tokens of the first text we make, and pass them to the generator for the second text. Then they will at least start with the same 'seed'.

## Detecting the watermark

How can we tell if text has been watermarked? There are two things we can do:

1. Simple: check the bias in the raw 'scores': the means of the g-values.
2. Tricky: make loads (hundreds or thousands) of pairs of watermarked and unwatermarked text plus their labels (0 or 1) and train a Bayesian classifier (`transformers.BayesianDetectorModel`) on the pairs.

**Both of these methods require us to know the key integers and the n-gram length that were used to watermark the text, and we need access to the tokenizer. However, we do not need to know anything about the model architecture or weights that produced the text.**

In this notebook, we do the simple thing (below). See [`Detect_With_Trained_Detector.ipynb`](./Detect_With_Trained_Detector.ipynb) for the classifier approach.

In [7]:
from transformers import SynthIDTextWatermarkLogitsProcessor

def compute_g_values(text, watermarking_config):
    logits_processor = SynthIDTextWatermarkLogitsProcessor(
        **watermarking_config.to_dict(),
        device=MODEL.device
    )
    input_ids = TOKENIZER(text, return_tensors="pt").input_ids
    return logits_processor.compute_g_values(input_ids)

g_values = compute_g_values(watermarked_text, watermarking_config)

These 'values' are actually small vectors of length $k$, where $k$ is the size of the keys vector, which is usually in the 8 to 16 range. Here's the first one (the first dimension is flat):

In [8]:
g_values[0, 0]

tensor([0, 0, 0])

There are $L - n + 1$ of these, where $L$ is the number of tokens in the output and $n$ is the length of the n-gram we specified in the watermarking configuration. (The n-grams are used together with the key to seed the randomness at each token generation step.)

In our case, $n = 5$

In [9]:
len(TOKENIZER(watermarked_text).input_ids), g_values.shape

(405, torch.Size([1, 401, 3]))

We are interested in the mean value all the g_values, because unwatermarked text has about equal numbers of 0's and 1's, while watermarking biases the little vectors towards "more ones".

So we take the mean, aka "score" or "g-score":

In [10]:
print(f"Mean score: {g_values.mean(dtype=float):.4f}")

Mean score: 0.6035


This is higher than 0.5 and so we suspect it is watermarked. But how sure can we be?

Since we know that the unadulterated score is Bernoulli(0.5) distributed, we can compute a p-value:

In [11]:
import scipy.stats as stats

def compute_p_value(g_values):
    """
    Statistically evaluate G-values against the 0.50 null hypothesis.
    """
    k_successes = int(g_values.sum().item())
    n_ngrams = g_values.numel()
    return stats.binomtest(k_successes, n_ngrams, p=0.5, alternative="greater").pvalue


p_value = compute_p_value(g_values)

print(f"Mean score: {g_values.mean(dtype=float):.3f}")
print(f"p-value:    {p_value:.3e}")

if p_value < 0.01:
    print("Verdict:    Watermarked")
else:
    print("Verdict:    Unwatermarked")

Mean score: 0.603
p-value:    3.604e-13
Verdict:    Watermarked


We interpret a p-value below 0.01 as meaning that there is less than a 1% chance that text occurred by random chance, if the null hypothesis is true (the null hypothesis is that the text is unwatermarked). The p-value is far below this, so we conclude that the text is almost certainly watermarked.

---

And check unwatermarked text...

In [12]:
g_values = compute_g_values(unwatermarked_text, watermarking_config)
p_value = compute_p_value(g_values)

print(f"Mean score: {g_values.mean(dtype=float):.3f}")
print(f"p-value:    {p_value:.3e}")

if p_value < 0.01:
    print("Verdict:    Watermarked")
else:
    print("Verdict:    Unwatermarked")

Mean score: 0.501
p-value:    4.880e-01
Verdict:    Unwatermarked


We interpret a p-value below 0.01 as meaning that there is less than a 1% chance the data are binamially distributed if the null hypothesis is true (i.e. if the text is not watermarked). The p-value is far below this, essentially zero.

---

## Visualize the affected tokens

A naive way to know which tokens were affected is to check the g_value means — anything over some threshold indicates that the token was probably not the most likely (argmax) token.

But we can get at the logits (the unscaled activations in the output layer) themselves and thus check if the selected token was in fact not the argmax token, then we know for sure. Getting the logits is an option in `AutoModelForCausalLM.generate()`

We'll use a longer set of keys so that the g_values threshold is easier.

In [13]:
import numpy as np

rng = np.random.default_rng(42)

watermarking_config = SynthIDTextWatermarkingConfig(
    keys=rng.integers(low=0, high=10_000, size=8), 
    ngram_len=5,
)

outputs = MODEL.generate(
    **inputs,
    max_new_tokens=300,
    return_dict_in_generate=True,
    output_logits=True,  # Returns raw model logits for generated tokens
    watermarking_config=watermarking_config,
)

Now we can get at full logit vectors (the activations of each token) via `outputs.logits`, and the usual output token id's via `outputs.sequences`. We want, in pseudocode:

    is_substituted = (mean(g_values) > 0.5)  & (selected_token != argmax_token)

We'll highlight affected tokens in red:

In [14]:
from IPython.display import HTML

# Remember the prompt is in the output... we need to account for
# this in order to align g_values, logits, and generated tokens.
# Yes, this did take me two hours to figure out.
prompt_length = inputs["input_ids"].shape[1]
generated_ids = outputs.sequences[0][prompt_length:]

html_parts = []

for g, step_logits, sample_id in zip(g_values[0], outputs.logits, generated_ids):

    # Decode the token we are appending.
    token = TOKENIZER.decode(sample_id, skip_special_tokens=False)

    # Get the raw greedy argmax token for this step.
    # step_logits shape is (batch_size, vocab_size) -> step_logits[0] is (vocab_size,)
    top_token_id = torch.argmax(step_logits[0]).item()
    argmax_token = TOKENIZER.decode(top_token_id, skip_special_tokens=False)

    # Clean display strings, argh.
    token_display = token.replace("\n", "<br>")
    argmax_display = argmax_token.replace('"', "&quot;").replace("\n", " ")

    # Check g-value.
    is_substituted = (sample_id != top_token_id) and (g.mean(dtype=float) > 0.50)

    if is_substituted:
        html_parts.append(
            f'<a style="background: Pink; margin: 0 1px; padding: 0 2px;" '
            f'title="{argmax_display}">{token_display}</a>'
        )
    else:
        html_parts.append(
            f'<span style="background: HoneyDew; margin: 0 2px; padding: 0 2px;">{token_display}</span>'
        )

display(
    HTML(
        f'<div style="font-family: monospace; line-height: 1.8; white-space: pre-wrap;">{"".join(html_parts)}</div>'
    )
)

The hover-over text is the argmax token. You might think we can undo the watermark by subbing this back in... but remember that **text generation is autoregressive** — we cannot replace the red tokens with their argmax tokens, because the following text depends on what was actually generated. The entire text snippet is a discrete, internally-consistent realization of the completion.

---
© 2026 Matt Hall / Equinor / CC BY / [AIA Ph CeNc Hin R v1.0](https://aiattribution.github.io/statements/AIA-Ph-CeNc-Hin-R-v1.0)